# B1 cluster-PGS validation in *All of Us*

External replication of the **B1 analysis** (multi-ancestry polygenic): associate **10 bNMF cluster polygenic scores (K1–K10)** with **T2D, CAD, and Synthetic (T2D & CAD)** in *All of Us*.

The statistical logic is **ported verbatim** from the Minerva pipeline (`scripts/b1_analysis/02_cluster_prs_association.R`) so results are directly comparable.

**Workflow**
1. *(terminal, run first)* `compute_prs_aou.sh` → `cluster_prs_all.tsv` (per-person `PRS_K1..PRS_K10`).
2. *(this notebook)* pull T2D/CAD phenotypes via OMOP, add covariates (age, age², sex, PC1–10, genetic ancestry), run individual + joint logistic models per ancestry stratum, draw forest plots.

**Cluster labels** · K1 Lpa · K2 Adiponectin · K3 Platelet · K4 SHBG · K5 Blood Pressure-Stature · K6 Metabolic · K7 Triglycerides-HDL · K8 ALP-LDL · K9 Obesity · K10 Glycemic

> **Differences from UKB B1 (deliberate):** the UKB genotyping-`Batch` covariate is dropped (no AoU analogue); ancestry uses AoU genetic-ancestry predictions, not UKB ADMIXTURE thresholds.

In [ ]:
## ============================ SETUP + PARAMETERS ============================
suppressMessages({
  library(data.table)
  library(pROC)
  library(ggplot2)
  library(bigrquery)
})

# --- AoU environment ---
WORKSPACE_BUCKET <- Sys.getenv("WORKSPACE_BUCKET")   # gs://...
WORKSPACE_CDR    <- Sys.getenv("WORKSPACE_CDR")      # BigQuery dataset for OMOP queries
GOOGLE_PROJECT   <- Sys.getenv("GOOGLE_PROJECT")     # billing project for bigrquery

# --- Inputs / outputs (edit if your paths differ) ---
PRS_BUCKET_PATH <- file.path(WORKSPACE_BUCKET, "b1_aou", "cluster_prs_all.tsv")  # from compute_prs_aou.sh
PRS_LOCAL       <- "cluster_prs_all.tsv"
# AoU genetic-ancestry predictions flat file (research_id, ancestry_pred, pca_features).
# In the AoU workbench this lives under the controlled datasets bucket; set it here:
ANCESTRY_FILE   <- Sys.getenv("AOU_ANCESTRY_FILE", unset = file.path(WORKSPACE_BUCKET, "b1_aou", "ancestry_preds.tsv"))
OUT_DIR         <- "b1_aou_results"; dir.create(OUT_DIR, showWarnings = FALSE)

# --- Analysis parameters (mirror B1) ---
CLUSTER_LABELS <- c(K1="Lpa", K2="Adiponectin", K3="Platelet", K4="SHBG",
                    K5="Blood Pressure-Stature", K6="Metabolic", K7="Triglycerides-HDL",
                    K8="ALP-LDL", K9="Obesity", K10="Glycemic")
ANCESTRY_STRATA <- c("afr", "amr", "eas", "eur", "mid", "sas")   # + a pooled "combined" run
COVAR_TERMS     <- c("age", "age2", "sex", paste0("PC", 1:10))   # NOTE: no UKB Batch term
OUTCOMES        <- c("T2D", "CAD", "Synthetic")
N_PCS           <- 10
MIN_CELL        <- 10   # skip a model if <10 cases or <10 controls

# --- Report all parameters before doing work (project rule) ---
cat("=== B1 AoU validation: parameters ===\n")
cat(sprintf("  WORKSPACE_BUCKET : %s\n", WORKSPACE_BUCKET))
cat(sprintf("  WORKSPACE_CDR    : %s\n", WORKSPACE_CDR))
cat(sprintf("  PRS file (bucket): %s\n", PRS_BUCKET_PATH))
cat(sprintf("  Ancestry file    : %s\n", ANCESTRY_FILE))
cat(sprintf("  Ancestry strata  : %s (+ combined)\n", paste(ANCESTRY_STRATA, collapse=", ")))
cat(sprintf("  Covariates       : %s\n", paste(COVAR_TERMS, collapse=" + ")))
cat(sprintf("  Outcomes         : %s\n", paste(OUTCOMES, collapse=", ")))
cat(sprintf("  Cluster PGS      : %s\n", paste(names(CLUSTER_LABELS), collapse=", ")))

# --- Copy the PRS table from the bucket and load it ---
system(sprintf("gsutil cp %s %s", PRS_BUCKET_PATH, PRS_LOCAL))
prs_all <- fread(PRS_LOCAL)
prs_cols <- grep("^PRS_K", colnames(prs_all), value = TRUE)
setnames(prs_all, "IID", "person_id", skip_absent = TRUE)
prs_all[, person_id := as.character(person_id)]
cat(sprintf("\nLoaded PRS: %d individuals, %d cluster PGS (%s)\n",
            nrow(prs_all), length(prs_cols), paste(prs_cols, collapse=", ")))
head(prs_all)

## 1 · Phenotypes — T2D & CAD via OMOP

**➜ FILL IN your concept sets below.** Paste the OMOP `condition_concept_id`s for T2D and CAD (e.g. the standard SNOMED concepts your Cohort Builder concept set expands to). A person is a **case** if they have ≥1 qualifying condition occurrence, otherwise a **control**. `Synthetic` follows the exact B1 rule: `T2D==1 & CAD==1` → case; `T2D==0 | CAD==0` → control; else `NA`.

In [ ]:
## =============================== PHENOTYPES ================================
## ====================== ### FILL IN ### concept sets =======================
# Paste the standard OMOP concept_ids that define each disease. You can include
# descendant concepts (recommended) by listing them here, or expand a Cohort
# Builder concept set to its concept_ids and paste them in.
T2D_CONCEPT_IDS <- c(
  ### FILL IN ###  e.g. 201826, 4193704, ...   (Type 2 diabetes mellitus)
)
CAD_CONCEPT_IDS <- c(
  ### FILL IN ###  e.g. 317576, 4185932, ...   (Coronary arteriosclerosis / CAD)
)
## ==========================================================================

stopifnot(length(T2D_CONCEPT_IDS) > 0, length(CAD_CONCEPT_IDS) > 0)

# Helper: pull the set of person_ids that have >=1 condition occurrence in a
# concept-id list (matches either the standard or source concept id).
get_cases <- function(concept_ids) {
  q <- sprintf("
    SELECT DISTINCT person_id
    FROM `%s.condition_occurrence`
    WHERE condition_concept_id IN (%s)
       OR condition_source_concept_id IN (%s)",
    WORKSPACE_CDR,
    paste(concept_ids, collapse = ","),
    paste(concept_ids, collapse = ","))
  tb <- bq_project_query(GOOGLE_PROJECT, q)
  as.character(bq_table_download(tb)$person_id)
}

t2d_cases <- get_cases(T2D_CONCEPT_IDS)
cad_cases <- get_cases(CAD_CONCEPT_IDS)
cat(sprintf("T2D cases: %d | CAD cases: %d\n", length(t2d_cases), length(cad_cases)))

# Case/control flags over the scored cohort (everyone with a PGS = denominator).
pheno <- prs_all[, .(person_id)]
pheno[, T2D := as.integer(person_id %in% t2d_cases)]
pheno[, CAD := as.integer(person_id %in% cad_cases)]
# Synthetic outcome (exact B1 rule)
pheno[, Synthetic := fifelse(T2D == 1 & CAD == 1, 1L,
                     fifelse(T2D == 0 | CAD == 0, 0L, NA_integer_))]
cat(sprintf("Synthetic cases: %d, controls: %d, NA: %d\n",
            sum(pheno$Synthetic == 1, na.rm=TRUE),
            sum(pheno$Synthetic == 0, na.rm=TRUE),
            sum(is.na(pheno$Synthetic))))
print(pheno[, .(T2D_cases=sum(T2D), CAD_cases=sum(CAD), N=.N)])

## 2 · Covariates & genetic ancestry

`age` (and `age²`), `sex` from the OMOP `person` table; `PC1–PC10` and the genetic-ancestry label from the AoU **ancestry-predictions** file (`pca_features` is a 16-element array — we keep the first 10 PCs to match B1).

In [ ]:
## =============================== COVARIATES ================================
REFERENCE_YEAR <- 2022   # CDR reference year for age = REFERENCE_YEAR - year_of_birth

# --- Demographics from the OMOP person table ---
demo_q <- sprintf("
  SELECT person_id, year_of_birth, sex_at_birth_concept_id
  FROM `%s.person`", WORKSPACE_CDR)
demo <- as.data.table(bq_table_download(bq_project_query(GOOGLE_PROJECT, demo_q)))
demo[, person_id := as.character(person_id)]
demo[, age  := REFERENCE_YEAR - as.integer(year_of_birth)]
demo[, age2 := age^2]
# sex: male = 1, female = 0 (AoU concept ids: 45880669 Male, 45878463 Female)
demo[, sex := fifelse(sex_at_birth_concept_id == 45880669, 1L,
              fifelse(sex_at_birth_concept_id == 45878463, 0L, NA_integer_))]
demo <- demo[, .(person_id, age, age2, sex)]

# --- Genetic ancestry + PCs from the AoU ancestry-predictions file ---
local_anc <- "ancestry_preds.tsv"
system(sprintf("gsutil cp %s %s", ANCESTRY_FILE, local_anc))
anc <- fread(local_anc)
# Normalize id + label column names across AoU file versions
setnames(anc, c("research_id"),  c("person_id"),     skip_absent = TRUE)
setnames(anc, c("ancestry_pred"),c("ancestry"),      skip_absent = TRUE)
setnames(anc, c("ancestry_pred_other"), c("ancestry"), skip_absent = TRUE)
anc[, person_id := as.character(person_id)]

# pca_features is a bracketed array string "[f1, f2, ... f16]" -> PC1..PC16
pc_mat <- tstrsplit(gsub("[\\[\\] ]", "", anc$pca_features), ",", fixed = FALSE)
for (i in seq_len(N_PCS)) anc[[paste0("PC", i)]] <- as.numeric(pc_mat[[i]])
anc <- anc[, c("person_id", "ancestry", paste0("PC", 1:N_PCS)), with = FALSE]

# --- Assemble the analysis table: PGS + phenotypes + covariates ---
dat <- merge(prs_all, pheno, by = "person_id")
dat <- merge(dat, demo,  by = "person_id", all.x = TRUE)
dat <- merge(dat, anc,   by = "person_id", all.x = TRUE)
cat(sprintf("Analysis table: %d individuals\n", nrow(dat)))
print(dat[, .N, by = ancestry][order(-N)])

## 3 · Association engine

Ported **verbatim** from `02_cluster_prs_association.R`: each cluster PGS is standardized (per-SD), then
- **individual** model: `outcome ~ PGS_Ki + age + age² + sex + PC1–10`
- **joint** model: `outcome ~ PGS_K1 + … + PGS_K10 + covariates`

Reports OR, 95% CI, p-value, **Nagelkerke R²**, and **AUC**; models with <10 cases or controls are skipped.

In [ ]:
## ========================== ASSOCIATION ENGINE ============================
## Ported from scripts/b1_analysis/02_cluster_prs_association.R (unchanged logic).

# --- Nagelkerke R-squared ---
nagelkerke_r2 <- function(full_model, null_model, n) {
  ll_full <- as.numeric(logLik(full_model))
  ll_null <- as.numeric(logLik(null_model))
  cox_snell <- 1 - exp((2 / n) * (ll_null - ll_full))
  max_r2 <- 1 - exp((2 / n) * ll_null)
  cox_snell / max_r2
}

# --- Run individual + joint regressions for one group ---
run_association <- function(data, group_name) {
  cat(sprintf("\n=== Running models on: %s (%d individuals) ===\n", group_name, nrow(data)))
  results_list <- list()

  for (outcome in OUTCOMES) {
    if (!outcome %in% colnames(data)) next

    # --- Individual models: outcome ~ PRS_K{i} + covariates ---
    for (prs_col in prs_cols) {
      model_data <- data[, c(outcome, prs_col, COVAR_TERMS), with = FALSE]
      model_data <- na.omit(model_data)
      n_cases <- sum(model_data[[outcome]] == 1); n_controls <- sum(model_data[[outcome]] == 0)
      if (n_cases < MIN_CELL || n_controls < MIN_CELL) next

      model_data[[prs_col]] <- scale(model_data[[prs_col]])[, 1]   # standardize PGS
      rhs <- paste(c(prs_col, COVAR_TERMS), collapse = " + ")
      formula_full <- as.formula(paste(outcome, "~", rhs))
      formula_null <- as.formula(paste(outcome, "~", paste(COVAR_TERMS, collapse = " + ")))

      fit_full <- tryCatch(glm(formula_full, data = model_data, family = binomial),
                           error = function(e) NULL)
      if (is.null(fit_full)) next
      fit_null <- glm(formula_null, data = model_data, family = binomial)

      prs_row <- summary(fit_full)$coefficients[prs_col, ]
      or <- exp(prs_row["Estimate"])
      ci_lower <- exp(prs_row["Estimate"] - 1.96 * prs_row["Std. Error"])
      ci_upper <- exp(prs_row["Estimate"] + 1.96 * prs_row["Std. Error"])
      p_val <- prs_row["Pr(>|z|)"]
      r2 <- nagelkerke_r2(fit_full, fit_null, nrow(model_data))
      auc_val <- tryCatch(as.numeric(auc(roc(model_data[[outcome]],
                          predict(fit_full, type = "response"), quiet = TRUE))),
                          error = function(e) NA_real_)

      results_list[[length(results_list) + 1]] <- data.table(
        group = group_name, outcome = outcome, model_type = "individual", predictor = prs_col,
        OR = round(or, 4), CI_lower = round(ci_lower, 4), CI_upper = round(ci_upper, 4),
        p_value = p_val, nagelkerke_r2 = round(r2, 6), AUC = round(auc_val, 4),
        n_cases = n_cases, n_controls = n_controls, n_total = n_cases + n_controls)
    }

    # --- Joint model: outcome ~ all PRS_K + covariates ---
    model_data <- data[, c(outcome, prs_cols, COVAR_TERMS), with = FALSE]
    model_data <- na.omit(model_data)
    n_cases <- sum(model_data[[outcome]] == 1); n_controls <- sum(model_data[[outcome]] == 0)
    if (n_cases < MIN_CELL || n_controls < MIN_CELL) next

    for (pc in prs_cols) model_data[[pc]] <- scale(model_data[[pc]])[, 1]
    rhs <- paste(c(prs_cols, COVAR_TERMS), collapse = " + ")
    formula_full <- as.formula(paste(outcome, "~", rhs))
    formula_null <- as.formula(paste(outcome, "~", paste(COVAR_TERMS, collapse = " + ")))
    fit_full <- tryCatch(glm(formula_full, data = model_data, family = binomial),
                         error = function(e) NULL)
    if (is.null(fit_full)) next
    fit_null <- glm(formula_null, data = model_data, family = binomial)
    coef_summ <- summary(fit_full)$coefficients
    r2 <- nagelkerke_r2(fit_full, fit_null, nrow(model_data))
    auc_val <- tryCatch(as.numeric(auc(roc(model_data[[outcome]],
                        predict(fit_full, type = "response"), quiet = TRUE))),
                        error = function(e) NA_real_)

    for (pc in prs_cols) {
      if (pc %in% rownames(coef_summ)) {
        prs_row <- coef_summ[pc, ]
        or <- exp(prs_row["Estimate"])
        ci_lower <- exp(prs_row["Estimate"] - 1.96 * prs_row["Std. Error"])
        ci_upper <- exp(prs_row["Estimate"] + 1.96 * prs_row["Std. Error"])
        p_val <- prs_row["Pr(>|z|)"]
      } else { or <- ci_lower <- ci_upper <- p_val <- NA_real_ }
      results_list[[length(results_list) + 1]] <- data.table(
        group = group_name, outcome = outcome, model_type = "joint", predictor = pc,
        OR = round(or, 4), CI_lower = round(ci_lower, 4), CI_upper = round(ci_upper, 4),
        p_value = p_val, nagelkerke_r2 = round(r2, 6), AUC = round(auc_val, 4),
        n_cases = n_cases, n_controls = n_controls, n_total = n_cases + n_controls)
    }
  }
  if (length(results_list) == 0) return(NULL)
  rbindlist(results_list)
}
cat("Association engine loaded.\n")

## 4 · Run across ancestry strata

Run every AoU genetic-ancestry stratum (`afr, amr, eas, eur, mid, sas`) separately, plus a pooled **combined** run (all participants, PCs adjust for ancestry). Results are written to `association_results_all.csv` and per-group CSVs, then copied back to the workspace bucket.

In [ ]:
## ========================= RUN ACROSS STRATA ==============================
groups <- c(setNames(lapply(ANCESTRY_STRATA, function(a) dat[ancestry == a]), ANCESTRY_STRATA),
            list(combined = dat))
for (g in names(groups)) cat(sprintf("  %s: %d individuals\n", g, nrow(groups[[g]])))

all_results <- rbindlist(lapply(names(groups), function(g) run_association(groups[[g]], g)),
                         fill = TRUE)

# --- Write results (combined + per-group), then copy to the bucket ---
combined_path <- file.path(OUT_DIR, "association_results_all.csv")
fwrite(all_results, combined_path)
for (g in unique(all_results$group)) {
  fwrite(all_results[group == g], file.path(OUT_DIR, sprintf("association_results_%s.csv", g)))
}
if (nzchar(WORKSPACE_BUCKET)) {
  system(sprintf("gsutil -m cp %s/*.csv %s/b1_aou/results/", OUT_DIR, WORKSPACE_BUCKET))
}
cat(sprintf("\nWrote %d rows -> %s (+ per-group CSVs, copied to bucket)\n",
            nrow(all_results), combined_path))

# Quick look: individual-model cluster PGS for each outcome in EUR
all_results[group == "eur" & model_type == "individual",
            .(predictor, outcome, OR, CI_lower, CI_upper, p_value, AUC)][order(outcome, predictor)]

## 5 · Forest plots

Per-cluster odds ratios (individual models, per-SD PGS) with 95% CIs, **coloured by outcome** (T2D / CAD / T2D and CAD) on a linear OR axis — styling ported verbatim from `03_b1_forest_plots.R`. Two figures: **`forest_eur_validation.png`** (EUR, the primary comparison to UKB B1 EUR) and **`forest_noneur_validation.png`** (non-EUR strata AFR/AMR/EAS/MID/SAS, faceted). Saved as PNGs and copied to the bucket. *(No genome-wide PGS in the AoU run, so B1's GW predictors and grey highlight band are omitted.)*

In [ ]:
## ============================= FOREST PLOTS ===============================
## Styling ported verbatim from scripts/b1_analysis/03_b1_forest_plots.R:
##   x = cluster predictor, y = OR per SD, colour = OUTCOME (not ancestry),
##   linear OR axis with a dashed reference line at OR = 1, dodge width 0.6.
## AoU has no genome-wide PGS, so B1's GW predictors and the grey highlight
## band are omitted; otherwise colours / labels / theme are identical.

# --- Label + colour conventions (identical to B1) ---
outcome_colors  <- c("Type 2 Diabetes" = "#E74C3C",
                     "Coronary Artery Disease" = "#3498DB",
                     "T2D and CAD" = "#9B59B6")
outcome_labels  <- c(T2D = "Type 2 Diabetes", CAD = "Coronary Artery Disease",
                     Synthetic = "T2D and CAD")
ancestry_labels <- c(afr = "AFR", amr = "AMR", eas = "EAS",
                     eur = "EUR", mid = "MID", sas = "SAS")

# --- Plot data: individual-model cluster PGS only ---
fp <- all_results[model_type == "individual" & !is.na(OR)]
fp[, predictor := sub("^PRS_", "", predictor)]                  # PRS_K1 -> K1
# Forest display order — matches scripts/b1_analysis/03_b1_forest_plots.R (desired_k_order):
# Glycemic, Obesity, SHBG, Adiponectin, Triglycerides-HDL, ALP-LDL, Metabolic, Platelet, Blood Pressure-Stature, Lpa
DESIRED_K_ORDER <- c("K10", "K9", "K4", "K2", "K7", "K8", "K6", "K3", "K5", "K1")
fp[, predictor := factor(CLUSTER_LABELS[predictor],
                         levels = unname(CLUSTER_LABELS[DESIRED_K_ORDER]))]  # forest display order
fp[, outcome_label := factor(outcome_labels[outcome], levels = outcome_labels)]
fp[, ancestry := ancestry_labels[group]]

make_forest <- function(d, title, point_size, facet = FALSE) {
  p <- ggplot(d, aes(x = predictor, y = OR, color = outcome_label)) +
    geom_hline(yintercept = 1, linetype = "dashed", color = "grey50", linewidth = 0.5) +
    geom_point(position = position_dodge(width = 0.6), size = point_size) +
    geom_errorbar(aes(ymin = CI_lower, ymax = CI_upper),
                  position = position_dodge(width = 0.6), width = 0.25, linewidth = 0.7) +
    scale_color_manual(values = outcome_colors, name = "Outcome") +
    labs(x = "Cluster PGS", y = "Odds Ratio per SD (95% CI)", title = title) +
    theme_minimal(base_size = 14) +
    theme(
      plot.background  = element_rect(fill = "white", color = NA),
      panel.background = element_rect(fill = "white", color = NA),
      plot.title   = element_text(face = "bold", size = 16),
      strip.text   = element_text(face = "bold", size = 14),
      axis.text.x  = element_text(angle = 45, hjust = 1, size = 11),
      axis.text.y  = element_text(size = 12),
      legend.position = "bottom",
      legend.title = element_text(face = "bold"),
      panel.grid.major.x = element_blank(),
      panel.grid.minor   = element_blank()
    )
  if (facet) p <- p + facet_wrap(~ ancestry, ncol = 1, scales = "free_y")
  p
}

# --- EUR validation (primary comparison to UKB B1 EUR) ---
eur_data <- fp[group == "eur"]
p_eur <- make_forest(eur_data, "AoU EUR: Cluster PGS → Disease", point_size = 3)
ggsave(file.path(OUT_DIR, "forest_eur_validation.png"), p_eur, width = 12, height = 7, dpi = 300)
cat("  Saved: forest_eur_validation.png\n")

# --- Non-EUR cross-ancestry portability (faceted by stratum) ---
noneur_strata <- setdiff(ANCESTRY_STRATA, "eur")
noneur_data   <- fp[group %in% noneur_strata]
noneur_data[, ancestry := factor(ancestry, levels = unname(ancestry_labels[noneur_strata]))]
p_noneur <- make_forest(noneur_data, "AoU Cross-Ancestry: Cluster PGS → Disease",
                        point_size = 2.5, facet = TRUE)
ggsave(file.path(OUT_DIR, "forest_noneur_validation.png"), p_noneur, width = 12, height = 14, dpi = 300)
cat("  Saved: forest_noneur_validation.png\n")

if (nzchar(WORKSPACE_BUCKET)) {
  system(sprintf("gsutil -m cp %s/*.png %s/b1_aou/results/", OUT_DIR, WORKSPACE_BUCKET))
}

options(repr.plot.width = 12, repr.plot.height = 7)
print(p_eur)

## 6 · Top-decile analysis

Top-quantile cluster-PGS associations, ported from `05_quantile_prs_analysis.R`. For each ancestry stratum (+ combined), each cluster PGS is dichotomized at its **within-group** top-q% threshold and the model `outcome ~ top_ind + age + age² + sex + PC1–10` gives the odds of disease in the top quantile vs. the rest. Quantiles **20 / 10 / 5 / 1%** are all computed (mirroring B1); the **top 10%** is the headline decile used in the forest plots below. Models with <10 cases or controls are skipped. *(AoU has no genome-wide PGS, so only the 10 cluster PGS are tested.)*

In [ ]:
## ====================== QUANTILE (TOP-DECILE) ANALYSIS ====================
## Ported from scripts/b1_analysis/05_quantile_prs_analysis.R (unchanged logic).
## For each group x cluster PGS x quantile, flag the top-q% (threshold computed
## WITHIN the group) and fit: outcome ~ top_ind + covariates. The top-10% rows
## drive the forest plots below; 20/5/1% are computed too for completeness.
## AoU has no genome-wide PGS, so only the cluster PGS are tested.

QUANTILE_FRACS <- c(0.20, 0.10, 0.05, 0.01)
QUANTILE_NAMES <- c("top_20pct", "top_10pct", "top_5pct", "top_1pct")

run_quantile <- function(data, group_name) {
  cat(sprintf("\n=== Quantile models on: %s (%d individuals) ===\n", group_name, nrow(data)))
  results_list <- list()

  for (prs_col in prs_cols) {
    for (qi in seq_along(QUANTILE_FRACS)) {
      frac  <- QUANTILE_FRACS[qi]
      qname <- QUANTILE_NAMES[qi]

      # Threshold within this group; top_ind = 1 if PGS in the top frac
      thresh_val <- quantile(data[[prs_col]], probs = 1 - frac, na.rm = TRUE)
      data[, top_ind := fifelse(get(prs_col) >= thresh_val, 1L, 0L)]
      n_top <- sum(data$top_ind == 1, na.rm = TRUE)

      for (outcome in OUTCOMES) {
        if (!outcome %in% colnames(data)) next
        model_data <- data[, c(outcome, "top_ind", COVAR_TERMS), with = FALSE]
        model_data <- na.omit(model_data)
        n_cases <- sum(model_data[[outcome]] == 1); n_controls <- sum(model_data[[outcome]] == 0)
        if (n_cases < MIN_CELL || n_controls < MIN_CELL) next

        n_cases_top    <- sum(model_data[[outcome]] == 1 & model_data$top_ind == 1)
        n_controls_top <- sum(model_data[[outcome]] == 0 & model_data$top_ind == 1)

        fit <- tryCatch(glm(as.formula(paste(outcome, "~ top_ind +",
                            paste(COVAR_TERMS, collapse = " + "))),
                            data = model_data, family = binomial),
                        error = function(e) NULL)
        if (is.null(fit)) next
        coef_summ <- summary(fit)$coefficients
        if (!"top_ind" %in% rownames(coef_summ)) next

        est <- coef_summ["top_ind", "Estimate"]
        se  <- coef_summ["top_ind", "Std. Error"]
        results_list[[length(results_list) + 1]] <- data.table(
          group = group_name, predictor = prs_col, outcome = outcome, quantile = qname,
          OR = round(exp(est), 4),
          CI_lower = round(exp(est - 1.96 * se), 4),
          CI_upper = round(exp(est + 1.96 * se), 4),
          p_value = coef_summ["top_ind", "Pr(>|z|)"],
          n_top = n_top, n_cases_top = n_cases_top, n_controls_top = n_controls_top)
      }
    }
  }
  if ("top_ind" %in% colnames(data)) data[, top_ind := NULL]
  if (length(results_list) == 0) return(NULL)
  rbindlist(results_list)
}

# Reuse the same group list (per-stratum + combined) as the per-SD models above
quantile_results <- rbindlist(lapply(names(groups), function(g) run_quantile(groups[[g]], g)),
                              fill = TRUE)
quantile_results[, cluster_label := CLUSTER_LABELS[sub("^PRS_", "", predictor)]]

# --- Write results (combined + per-group), then copy to the bucket ---
quant_path <- file.path(OUT_DIR, "quantile_prs_associations.csv")
fwrite(quantile_results, quant_path)
for (g in unique(quantile_results$group)) {
  fwrite(quantile_results[group == g],
         file.path(OUT_DIR, sprintf("quantile_prs_associations_%s.csv", g)))
}
if (nzchar(WORKSPACE_BUCKET)) {
  system(sprintf("gsutil -m cp %s/quantile_prs_associations*.csv %s/b1_aou/results/",
                 OUT_DIR, WORKSPACE_BUCKET))
}
cat(sprintf("\nWrote %d rows -> %s (+ per-group CSVs, copied to bucket)\n",
            nrow(quantile_results), quant_path))

# Quick look: EUR top-10% ORs per cluster PGS and outcome
quantile_results[group == "eur" & quantile == "top_10pct",
                 .(cluster_label, outcome, OR, CI_lower, CI_upper, p_value,
                   n_top, n_cases_top)][order(outcome, cluster_label)]

### 6.1 · Top-decile forest plots

Top-10% odds ratios with 95% CIs, **coloured by outcome** — styling ported verbatim from `05b_quantile_forest_plots.R`. Two figures: **`forest_eur_top10pct.png`** (EUR, the primary comparison to UKB B1 EUR) and **`forest_noneur_top10pct.png`** (non-EUR strata AFR/AMR/EAS/MID/SAS, faceted). Saved as PNGs and copied to the bucket. *(No genome-wide PGS in the AoU run, so B1's GW predictors and grey highlight band are omitted.)*

In [ ]:
## ====================== TOP-DECILE FOREST PLOTS ===========================
## Styling ported verbatim from scripts/b1_analysis/05b_quantile_forest_plots.R:
##   x = cluster predictor, y = OR in top decile, colour = OUTCOME,
##   linear OR axis with a dashed reference line at OR = 1, dodge width 0.6.
## AoU has no genome-wide PGS, so B1's GW predictors and the grey highlight
## band are omitted; otherwise colours / labels / theme are identical.

# --- Top-decile plot data: cluster PGS only (no GW in AoU) ---
top10 <- quantile_results[quantile == "top_10pct" & !is.na(OR)]
top10[, predictor := sub("^PRS_", "", predictor)]               # PRS_K1 -> K1
# Forest display order — matches 05b_quantile_forest_plots.R (desired_k_order):
# Glycemic, Obesity, SHBG, Adiponectin, Triglycerides-HDL, ALP-LDL, Metabolic, Platelet, Blood Pressure-Stature, Lpa
top10[, predictor := factor(CLUSTER_LABELS[predictor],
                            levels = unname(CLUSTER_LABELS[DESIRED_K_ORDER]))]
top10[, outcome_label := factor(outcome_labels[outcome], levels = outcome_labels)]
top10[, ancestry := ancestry_labels[group]]

make_top10_forest <- function(d, title, point_size, facet = FALSE) {
  p <- ggplot(d, aes(x = predictor, y = OR, color = outcome_label)) +
    geom_hline(yintercept = 1, linetype = "dashed", color = "grey50", linewidth = 0.5) +
    geom_point(position = position_dodge(width = 0.6), size = point_size) +
    geom_errorbar(aes(ymin = CI_lower, ymax = CI_upper),
                  position = position_dodge(width = 0.6), width = 0.25, linewidth = 0.7) +
    scale_color_manual(values = outcome_colors, name = "Outcome") +
    labs(x = "Cluster PGS", y = "OR in Top Decile (95% CI)", title = title) +
    theme_minimal(base_size = 14) +
    theme(
      plot.background  = element_rect(fill = "white", color = NA),
      panel.background = element_rect(fill = "white", color = NA),
      plot.title   = element_text(face = "bold", size = 16),
      strip.text   = element_text(face = "bold", size = 14),
      axis.text.x  = element_text(angle = 45, hjust = 1, size = 11),
      axis.text.y  = element_text(size = 12),
      legend.position = "bottom",
      legend.title = element_text(face = "bold"),
      panel.grid.major.x = element_blank(),
      panel.grid.minor   = element_blank()
    )
  if (facet) p <- p + facet_wrap(~ ancestry, ncol = 1, scales = "free_y")
  p
}

# --- EUR validation (primary comparison to UKB B1 EUR) ---
eur_top10 <- top10[group == "eur"]
p_eur_top10 <- make_top10_forest(eur_top10, "AoU EUR: Top 10% Cluster PGS → Disease",
                                 point_size = 3)
ggsave(file.path(OUT_DIR, "forest_eur_top10pct.png"), p_eur_top10, width = 12, height = 7, dpi = 300)
cat("  Saved: forest_eur_top10pct.png\n")

# --- Non-EUR cross-ancestry portability (faceted by stratum) ---
noneur_top10 <- top10[group %in% noneur_strata]
noneur_top10[, ancestry := factor(ancestry, levels = unname(ancestry_labels[noneur_strata]))]
p_noneur_top10 <- make_top10_forest(noneur_top10, "AoU Cross-Ancestry: Top 10% Cluster PGS → Disease",
                                    point_size = 2.5, facet = TRUE)
ggsave(file.path(OUT_DIR, "forest_noneur_top10pct.png"), p_noneur_top10, width = 12, height = 14, dpi = 300)
cat("  Saved: forest_noneur_top10pct.png\n")

if (nzchar(WORKSPACE_BUCKET)) {
  system(sprintf("gsutil -m cp %s/*.png %s/b1_aou/results/", OUT_DIR, WORKSPACE_BUCKET))
}

options(repr.plot.width = 12, repr.plot.height = 7)
print(p_eur_top10)